In [12]:
!pip install pandas numpy scikit-learn matplotlib pyarrow -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\9901359\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
# setting up environment for ETL assessment
import pandas as pd
import numpy as np
import os,json,csv,time,random,logging
from datetime import date,datetime,timedelta

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO,format="%(asctime)s [%(levelname)s] %(message)s")

In [14]:
# generating and loading the dataset

PRODUCTS = ['Laptop','Monitor','Keyboard','Mouse','Headset','Webcam','USB Hub','Desk Lamp','Speaker','Tablet']
REGIONS = ['North','South','East','West']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': PRODUCTS[i % len(PRODUCTS)].split()[0] if i < 50 else random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })

df = pd.DataFrame(records)
df['revenue'] = (df['quantity']*df['unit_price']).round(2)
print(df.shape[0])
print(df.shape[1])
df.head()

10000
9


,order_id,product,category,region,quantity,unit_price,order_date,status,revenue
0,ORD-00000,Monitor,Laptop,North,48,282.27,2024-04-24,completed,13548.96
1,ORD-00001,Monitor,Monitor,North,38,427.69,2024-01-16,completed,16252.22
2,ORD-00002,Mouse,Keyboard,North,36,206.84,2024-11-28,pending,7446.24
3,ORD-00003,USB Hub,Mouse,South,29,593.36,2024-01-04,pending,17207.44
4,ORD-00004,Keyboard,Headset,West,22,285.08,2024-04-20,refunded,6271.76


In [15]:
print('=' * 60)
print('DATA PROFILE: W2 Assessment: ETL Engineering')
print('=' * 60)
print(f'\nShape: {df.shape}')
print(f'\nColumn Types:\n{df.dtypes}')
print(f'\nNull Counts:\n{df.isnull().sum()}')
print(f'\nDuplicate Rows: {df.duplicated().sum()}')
print(f'\nNumeric Summary:\n{df.describe()}')
print(f'\nUnique Values:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()} unique')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

DATA PROFILE: W2 Assessment: ETL Engineering

Shape: (10000, 9)

Column Types:
order_id          str
product           str
category          str
region            str
quantity        int64
unit_price    float64
order_date        str
status            str
revenue       float64
dtype: object

Null Counts:
order_id      0
product       0
category      0
region        0
quantity      0
unit_price    0
order_date    0
status        0
revenue       0
dtype: int64

Duplicate Rows: 0

Numeric Summary:
           quantity    unit_price       revenue
count  10000.000000  10000.000000  10000.000000
mean      25.140600    503.807532  12629.256019
std       14.580209    287.654131  11033.664281
min        1.000000     10.050000     16.440000
25%       12.000000    254.082500   3529.290000
50%       25.000000    502.315000   9347.180000
75%       38.000000    751.495000  19358.000000
max       50.000000    999.970000  49685.500000

Unique Values:
  order_id: 10000 unique
  product: 10 unique
  categ

In [16]:
# Cell 4: Core Implementation — W2 Assessment: ETL Engineering

import sqlite3
import os
import time

# ============================================================
# End-to-End ETL Assessment: CSV -> Validate -> Transform -> SQLite
# ============================================================

logging.info('Starting W2 Assessment: ETL Engineering processing...')

# --- Stage 1: Extract (from our generated DataFrame) ---
raw_df = df.copy()
print(f'EXTRACT: {len(raw_df)} rows loaded')

# --- Stage 2: Validate with quality gate ---
quality_report = {}
quality_report['total_rows'] = len(raw_df)
quality_report['null_pcts'] = raw_df.isnull().mean().to_dict()
quality_report['duplicate_ids'] = raw_df['order_id'].duplicated().sum()

# Quality gate: fail if >5% nulls in any column
max_null = raw_df.isnull().mean().max()
quality_report['max_null_pct'] = round(max_null, 4)
quality_report['quality_gate'] = 'PASS' if max_null < 0.05 else 'FAIL'
print(f'VALIDATE: Quality gate = {quality_report["quality_gate"]} (max null: {max_null:.2%})')

# --- Stage 3: Transform ---
etl_df = raw_df.copy()
etl_df['order_date'] = pd.to_datetime(etl_df['order_date'])
etl_df['year_month'] = etl_df['order_date'].dt.to_period('M').astype(str)
etl_df['quarter'] = etl_df['order_date'].dt.quarter
etl_df['revenue'] = (etl_df['quantity'] * etl_df['unit_price']).round(2)
etl_df['region'] = etl_df['region'].str.title()

# Create summary aggregation
summary = etl_df.groupby(['region', 'year_month']).agg(
    orders=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean')
).round(2).reset_index()
print(f'TRANSFORM: Created {len(summary)} summary rows')

# --- Stage 4: Load into SQLite ---
db_path = '/tmp/etl_assessment.db'
conn = sqlite3.connect(db_path)
etl_df.to_sql('orders_clean', conn, index=False, if_exists='replace')
summary.to_sql('revenue_summary', conn, index=False, if_exists='replace')

# Verify load
loaded_count = pd.read_sql('SELECT COUNT(*) AS cnt FROM orders_clean', conn).iloc[0, 0]
summary_count = pd.read_sql('SELECT COUNT(*) AS cnt FROM revenue_summary', conn).iloc[0, 0]
print(f'LOAD: {loaded_count} detail rows + {summary_count} summary rows -> SQLite')

# --- Stage 5: Run analytical queries on warehouse ---
print('\n=== Warehouse Query: Top regions by revenue ===')
result = pd.read_sql('''
    SELECT region, SUM(total_revenue) AS revenue, SUM(orders) AS order_count
    FROM revenue_summary GROUP BY region ORDER BY revenue DESC
''', conn)
print(result)

conn.close()
print(f'\nQuality Report: {quality_report}')
print('-- W2 Assessment: ETL Engineering implementation complete')

2026-03-27 11:17:09,998 [INFO] Starting W2 Assessment: ETL Engineering processing...


EXTRACT: 10000 rows loaded
VALIDATE: Quality gate = PASS (max null: 0.00%)
TRANSFORM: Created 48 summary rows
LOAD: 10000 detail rows + 48 summary rows -> SQLite

=== Warehouse Query: Top regions by revenue ===
  region      revenue  order_count
0   West  32704052.11         2525
1   East  31929463.06         2558
2  South  30902956.65         2483
3  North  30756088.37         2434

Quality Report: {'total_rows': 10000, 'null_pcts': {'order_id': 0.0, 'product': 0.0, 'category': 0.0, 'region': 0.0, 'quantity': 0.0, 'unit_price': 0.0, 'order_date': 0.0, 'status': 0.0, 'revenue': 0.0}, 'duplicate_ids': np.int64(0), 'max_null_pct': np.float64(0.0), 'quality_gate': 'PASS'}
-- W2 Assessment: ETL Engineering implementation complete


In [17]:
print('=' * 60)
print('RESULTS: W2 Assessment: ETL Engineering')
print('=' * 60)
print(f'Input rows: {len(df)}')
print(f'Processing complete')

# Display key metrics
print(f'\nRevenue by Region:')
print(df.groupby('region')['revenue'].sum().sort_values(ascending=False))

RESULTS: W2 Assessment: ETL Engineering
Input rows: 10000
Processing complete

Revenue by Region:
region
West     32704052.11
East     31929463.06
South    30902956.65
North    30756088.37
Name: revenue, dtype: float64


In [18]:
# Cell 6: Self-Check — ETL Assessment
# Run this cell to verify your work before clicking "Run Tests"
print('=' * 50)
print('SELF-CHECK — ETL Assessment')
print('=' * 50)

checks = {
    'DataFrame exists and is not empty': len(df) > 0,
    'Has at least 2 columns': len(df.columns) >= 2,
    'No fully-null columns': df.isnull().mean().max() < 0.5,
    'Has at least 10 rows': len(df) >= 10,
    'Data types look valid': df.dtypes is not None,
}

passed = sum(1 for v in checks.values() if v)
for name, ok in checks.items():
    print(f'  {"PASS" if ok else "FAIL"}: {name}')

print(f'\n{passed}/{len(checks)} self-checks passed')
if passed == len(checks):
    print('\nAll good! Click "Run Tests" to submit for official validation.')
else:
    print('\nSome checks failed. Fix your code, then click "Run Tests".')


SELF-CHECK — ETL Assessment
  PASS: DataFrame exists and is not empty
  PASS: Has at least 2 columns
  PASS: No fully-null columns
  PASS: Has at least 10 rows
  PASS: Data types look valid

5/5 self-checks passed

All good! Click "Run Tests" to submit for official validation.
